# exp01 — Trích landmark từ Multi-VSL 200 (view chính diện)

**Mở trên Colab:** https://colab.research.google.com/github/minKasent/signbridge/blob/main/training/experiments/exp01_extract_landmarks.ipynb

Biến ~5.900 video (subset 200 ký hiệu, chỉ view chính diện, split đã tách theo người ký 20/4/4)
thành mảng landmark 144 chiều — dùng **đúng file model .task mà browser dùng** để phân bố
train/deploy đồng nhất.

- **Runtime**: CPU là đủ (MediaPipe Tasks Python chạy CPU) — không cần bật GPU.
- **Thời gian**: ~3-5 giờ cho split train. **Resume-safe**: Colab rớt phiên thì mở lại,
  chạy lại từ đầu — file đã trích sẽ tự bỏ qua.
- Kết quả lưu thẳng vào Drive: `MyDrive/datn/landmarks/` (~vài trăm MB), không mất khi hết phiên.

In [ ]:
# Cell 1 — Cài đặt + tải code (~1 phút)
%pip install -q mediapipe opencv-python tqdm numpy
!git clone -q https://github.com/minKasent/signbridge.git /content/signbridge 2>/dev/null || (cd /content/signbridge && git pull -q)
!git clone -q https://github.com/Etdihatthoc/Multi-VSL_WACV_2025.git /content/multivsl 2>/dev/null || true

import mediapipe, os
print("MediaPipe:", mediapipe.__version__)
print("Model .task:", os.listdir("/content/signbridge/apps/web/public/models"))
print("CSV split:", sorted(os.listdir("/content/multivsl/data/label_1_200"))[:4], "...")

In [ ]:
# Cell 2 — Mount Drive + kiểm tra dataset
from google.colab import drive
drive.mount("/content/drive")

import csv, os, random

VIDEOS_DIR = "/content/drive/MyDrive/datn/Videos_demo"
CSV_DIR = "/content/multivsl/data/label_1_200"

assert os.path.isdir(VIDEOS_DIR), f"Không thấy {VIDEOS_DIR} — kiểm tra lối tắt Drive (SETUP-KEYS.md mục 7)"

# Liệt kê folder (folder lớn — có thể mất 1-3 phút lần đầu)
print("Đang liệt kê folder Drive…")
all_files = set(os.listdir(VIDEOS_DIR))
mp4 = [f for f in all_files if f.endswith(".mp4")]
khac = [f for f in all_files if not f.endswith(".mp4")]
print(f"Tổng: {len(all_files)} file — {len(mp4)} video, {len(khac)} file khác")
if khac:
    print("File KHÔNG phải video (có thể là file từ vựng của tác giả — mở xem thử!):", khac[:20])

# Đối chiếu CSV ↔ folder: subset center cần bao nhiêu video và folder có đủ không?
for split in ["train", "val", "test"]:
    with open(f"{CSV_DIR}/{split}_1_200_center_ord1.csv", newline="") as f:
        names = [r["name"] for r in csv.DictReader(f)]
    have = sum(1 for n in names if n in all_files)
    print(f"{split}: cần {len(names)} video — có sẵn {have} ({have*100//len(names)}%)")

In [ ]:
# Cell 3 — Trích landmark (LÂU: ~3-5h cho train; chạy lại được nếu rớt phiên)
MODELS = "/content/signbridge/apps/web/public/models"
OUT = "/content/drive/MyDrive/datn/landmarks"

for split in ["val", "test", "train"]:  # val/test trước (nhanh) để thấy kết quả sớm
    print(f"===== {split} =====")
    !python /content/signbridge/training/preprocess/extract_landmarks.py \
        --videos "{VIDEOS_DIR}" \
        --csv "{CSV_DIR}/{split}_1_200_center_ord1.csv" \
        --models "{MODELS}" \
        --out "{OUT}/{split}"

In [ ]:
# Cell 4 — Thống kê kết quả
import numpy as np, glob, os

OUT = "/content/drive/MyDrive/datn/landmarks"
for split in ["train", "val", "test"]:
    files = glob.glob(f"{OUT}/{split}/*/*.npy")
    labels = set(os.path.basename(os.path.dirname(f)) for f in files)
    print(f"{split}: {len(files)} chuỗi, {len(labels)} lớp")
    if files:
        sample = np.load(files[0])
        print(f"   ví dụ {os.path.basename(files[0])}: shape={sample.shape} (T, 144), dtype={sample.dtype}")

print("\nXong → bước kế: exp02_train_baseline.ipynb (train Transformer trên các file này)")

## Ghi chú

- Nhãn trong CSV là **số 0-198** — repo tác giả không kèm file ánh xạ sang tên ký hiệu tiếng Việt.
  Cell 2 in ra các file không phải video trong folder Drive: nếu có file .txt/.xlsx thì rất có thể
  là danh sách từ vựng — báo lại để cập nhật từ điển. Không có thì email tác giả (có mẫu sẵn khi cần).
- Nếu Drive báo quota tải: chạy lại sau vài giờ — script tự bỏ qua file đã trích.
- Muốn nhanh hơn: Runtime → Change runtime type → chọn máy nhiều CPU hơn (nếu có Colab Pro),
  hoặc cứ để chạy qua đêm — miễn phí.